In [3]:
library(tidyverse)
library(tsibble)
library(fable)
library(fabletools)
library(feasts)
library(lubridate)

options(repr.plot.width = 12, repr.plot.height = 5, repr.plot.res = 150)

Warning message:
"package 'tidyverse' was built under R version 4.5.3"
Warning message:
"package 'ggplot2' was built under R version 4.5.3"
Warning message:
"package 'tibble' was built under R version 4.5.3"
Warning message:
"package 'tidyr' was built under R version 4.5.3"
Warning message:
"package 'readr' was built under R version 4.5.3"
Warning message:
"package 'purrr' was built under R version 4.5.3"
Warning message:
"package 'dplyr' was built under R version 4.5.3"
Warning message:
"package 'stringr' was built under R version 4.5.2"
Warning message:
"package 'forcats' was built under R version 4.5.2"
Warning message:
"package 'lubridate' was built under R version 4.5.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ──────────────────────────────────────

In [4]:
# Load pre-cleaned data (produced by data_cleaning.ipynb)
df_lagged  <- readr::read_rds("fluview_clean/df_lagged.rds")
train_full <- readr::read_rds("fluview_clean/train_full.rds")
holdout    <- readr::read_rds("fluview_clean/holdout.rds")
n_train_init <- nrow(train_full)

cat("Rows:", nrow(df_lagged), "\n")
cat("Holdout range:", as.character(min(holdout$week_start)),
    "to", as.character(max(holdout$week_start)), "\n")

Rows: 1198 


Holdout range: 2023 W01 to 2024 W52 


In [18]:
rolling_rmse <- function(data, model_expr, model_name, min_train = n_train_init) {

  stretched <- data %>%
    stretch_tsibble(.init = min_train, .step = 1)

  rolling_fits <- rlang::inject(
    model(stretched, mod = !!model_expr)
  )

  rolling_fc <- rolling_fits %>%
    forecast(h = 1)

  actuals <- data %>%
    as_tibble() %>%
    select(week_start, actual = percent_weighted_ili)

  results <- rolling_fc %>%
    as_tibble() %>%
    left_join(actuals, by = "week_start") %>%
    mutate(
      model    = model_name,
      forecast = .mean,
      sq_error = (actual - forecast)^2
    ) %>%
    select(model, week_start, actual, forecast, sq_error)

  results
}

rolling_arimax_base <- function(data, covariate_cols,   
                                order    = c(1, 1, 0),
                                seasonal = list(order = c(1, 1, 0), period = 52),
                                min_train = n_train_init,
                                method = "CSS") {   # ← add this
  df_tb <- as_tibble(data)
  n     <- nrow(df_tb)
  forecasts <- rep(NA_real_, n)

  for (t in (min_train + 1):n) {
    train_y <- df_tb$percent_weighted_ili[1:(t - 1)]
    train_x <- as.matrix(df_tb[1:(t - 1), covariate_cols])
    new_x   <- as.matrix(df_tb[t, covariate_cols])

    fit <- tryCatch(
      arima(train_y, order = order, seasonal = seasonal, 
            xreg = train_x, method = method),   # ← pass through here
      error = function(e) tryCatch(
        arima(train_y, order = order, seasonal = seasonal,
              xreg = train_x, method = "CSS"),  # ← CSS as fallback
        error = function(e) NULL
      )
    )
    if (!is.null(fit)) {
      forecasts[t] <- predict(fit, n.ahead = 1, newxreg = new_x)$pred[1]
    }
  }

  tibble(
    model      = paste(covariate_cols, collapse = " + "),
    week_start = df_tb$week_start,
    actual     = df_tb$percent_weighted_ili,
    forecast   = forecasts
  ) %>%
    filter(!is.na(forecast)) %>%
    mutate(sq_error = (actual - forecast)^2)
}

In [19]:
res_arimax5 <- rolling_arimax_base(
  df_lagged,
  covariate_cols = "d_age_65_lag1",
  min_train = n_train_init,
  method = "CSS"  # ← you'll need to add this arg to rolling_arimax_base
)

In [20]:
res_arimax2 <- rolling_arimax_base(df_lagged, covariate_cols = "d_age_0_4_lag1", min_train = n_train_init, method = "CSS")
res_arimax3 <- rolling_arimax_base(df_lagged, covariate_cols = "d_age_25_49_lag1", min_train = n_train_init, method = "CSS")

In [21]:
res_arimax <- rolling_arimax_base(df_lagged, covariate_col = "d_age_5_24_lag1", min_train = n_train_init, method = "CSS")
res_arimax4 <- rolling_arimax_base(df_lagged, covariate_col = "d_age_50_64_lag1", min_train = n_train_init, method = "CSS")

In [22]:
args(rolling_arimax_base)

function (data, covariate_cols, order = c(1, 1, 0), seasonal = list(order = c(1, 
    1, 0), period = 52), min_train = n_train_init, method = "CSS") 
NULL

In [33]:
all_covariates <- c("d_age_0_4_lag1", "d_age_5_24_lag1",
                    "d_age_25_49_lag1", "d_age_50_64_lag1", "d_age_65_lag1")

res_arimax_all <- rolling_arimax_base(df_lagged, covariate_cols = all_covariates, min_train = n_train_init, method = "CSS")

In [35]:
young_covariates <- c("d_age_0_4_lag1", "d_age_5_24_lag1")

res_arimax_top2 <- rolling_arimax_base(df_lagged, covariate_cols = young_covariates, min_train = n_train_init, method = "CSS")

In [37]:
top3_covariates <- c("d_age_0_4_lag1", "d_age_5_24_lag1", "d_age_65_lag1")

res_arimax_top3 <- rolling_arimax_base(df_lagged, covariate_cols = top3_covariates, min_train = n_train_init, method = "CSS")

In [38]:
# Label each result with its covariate before binding
arimax_perturb_table <- bind_rows(
  res_arimax  %>% mutate(model = "ARIMAX — age_5_24 only"),
  res_arimax2 %>% mutate(model = "ARIMAX — age_0_4 only"),
  res_arimax3 %>% mutate(model = "ARIMAX — age_25_49 only"),
  res_arimax4 %>% mutate(model = "ARIMAX — age_50_64 only"),
  res_arimax5 %>% mutate(model = "ARIMAX — age_65 only"),
  res_arimax_all %>% mutate(model = "ARIMAX — all covariates"),
  res_arimax_top2 %>% mutate(model = "ARIMAX — top 2 covariates"),
  res_arimax_top3 %>% mutate(model = "ARIMAX — top 3 covariates")
) %>%
  filter(week_start %in% holdout$week_start) %>%
  group_by(model) %>%
  summarise(
    RMSE = round(sqrt(mean(sq_error, na.rm = TRUE)), 3),
    n    = sum(!is.na(sq_error)),
    .groups = "drop"
  ) %>%
  arrange(RMSE)

print(arimax_perturb_table)

# A tibble: 8 × 3
  model                      RMSE     n
  <chr>                     <dbl> <int>
1 ARIMAX — age_5_24 only    0.25    104
2 ARIMAX — top 2 covariates 0.254   104
3 ARIMAX — top 3 covariates 0.254   104
4 ARIMAX — age_0_4 only     0.258   104
5 ARIMAX — age_65 only      0.261   104
6 ARIMAX — age_50_64 only   0.262   104
7 ARIMAX — age_25_49 only   0.27    104
8 ARIMAX — all covariates   0.29    104


In [12]:
# install.packages("forecast")
library(forecast)

Warning message:
"package 'forecast' was built under R version 4.5.3"


In [28]:
# Use your best covariate, e.g., age_5_24
train_x <- as.matrix(train_full[, "d_age_5_24_lag1"])
train_y <- train_full$percent_weighted_ili

auto_fit <- auto.arima(
  train_y,
  xreg        = train_x,
  seasonal    = TRUE,
  max.p       = 3, max.q   = 3,
  max.P       = 2, max.Q   = 2,
  d = 1,
  stepwise    = FALSE,      # exhaustive search, slower but better
  approximation = FALSE,    # exact likelihood
  ic          = "aic"       # or "bic"
)

summary(auto_fit)

Series: train_y 
Regression with ARIMA(3,1,2) errors 

Coefficients:
          ar1      ar2     ar3     ma1     ma2   drift  d_age_5_24_lag1
      -0.3609  -0.4081  0.3530  0.8227  0.8424  0.0027                0
s.e.   0.1007   0.0946  0.0635  0.0925  0.0958  0.0001                0

sigma^2 = 0.09097:  log likelihood = -237.49
AIC=490.97   AICc=491.1   BIC=530.94

Training set error measures:
                        ME      RMSE       MAE MPE MAPE      MASE         ACF1
Training set -0.0001766534 0.3005112 0.1720054 NaN  Inf 0.8358049 4.700657e-05

In [29]:
# TODO: TO RUN AT LIBRARY
arimax_orders <- list(
  # New auto.arima baseline: ARIMA(1,1,0) — non-seasonal
  list(name = "ARIMAX(1,1,0)(0,0,0)",
       order = c(1,1,0), seasonal = list(order=c(0,0,0), period=52)),

  # Neighbors: vary q upward from baseline
  list(name = "ARIMAX(1,1,1)(0,0,0)",
       order = c(1,1,1), seasonal = list(order=c(0,0,0), period=52)),
  list(name = "ARIMAX(1,1,2)(0,0,0)",
       order = c(1,1,2), seasonal = list(order=c(0,0,0), period=52)),

  # Neighbors: vary p upward from baseline
  list(name = "ARIMAX(2,1,0)(0,0,0)",
       order = c(2,1,0), seasonal = list(order=c(0,0,0), period=52)),
  list(name = "ARIMAX(2,1,1)(0,0,0)",
       order = c(2,1,1), seasonal = list(order=c(0,0,0), period=52)),

  # Light seasonal variants — D=1 forced as in your auto.arima call
  list(name = "ARIMAX(1,1,0)(0,1,0)[52]",
       order = c(1,1,0), seasonal = list(order=c(0,1,0), period=52)),
  list(name = "ARIMAX(1,1,1)(0,1,0)[52]",
       order = c(1,1,1), seasonal = list(order=c(0,1,0), period=52)),
  list(name = "ARIMAX(1,1,0)(1,1,0)[52] - baseline",
       order = c(1,1,0), seasonal = list(order=c(1,1,0), period=52))
)

arimax_perturb_results <- purrr::map_dfr(arimax_orders, function(spec) {
  res <- rolling_arimax_base(
    df_lagged,
    covariate_cols = "d_age_5_24_lag1",
    order    = spec$order,
    seasonal = spec$seasonal,
    method = "CSS"
  )
  res %>%
    filter(week_start %in% holdout$week_start) %>%
    summarise(
      model = spec$name,
      RMSE  = round(sqrt(mean(sq_error, na.rm = TRUE)), 4),
      n     = sum(!is.na(sq_error))
    )
})

arimax_perturb_results %>% arrange(RMSE)

model,RMSE,n
<chr>,<dbl>,<int>
"ARIMAX(1,1,0)(1,1,0)[52] - baseline",0.2500,104
"ARIMAX(1,1,0)(0,0,0)",0.2792,104
"ARIMAX(1,1,1)(0,0,0)",0.2803,104
"ARIMAX(2,1,0)(0,0,0)",0.2805,104
"ARIMAX(1,1,2)(0,0,0)",0.2806,104
"ARIMAX(2,1,1)(0,0,0)",0.2816,104
"ARIMAX(1,1,0)(0,1,0)[52]",0.2966,104
"ARIMAX(1,1,1)(0,1,0)[52]",0.3025,104
